# ViT (Vision Transformer) — 从 Transformer Encoder 改造而来

**改造思路**：
- SelfAttention / MultiHeadAttention / FeedForward / EncoderLayer → **直接复用**
- nn.Embedding + 正弦位置编码 → **PatchEmbedding + 可学习位置编码**
- 整个 Decoder → **删除**
- 新增 CLS token 做分类

对比文件：`_Transformer.ipynb`

In [ ]:
import torch
from torch import nn
import math

## 1. 基础组件 — 核心计算与 Transformer 一致，架构改为 Pre-LN

### 自注意力机制（与 Transformer 完全一致）

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, Q, K, V, mask=None):
        # Q, K, V: (B, n_heads, N, d_k)   N = 1 + n_patches (CLS + patches)
        d_k = Q.size(-1)
        # (B, n_heads, N, d_k) × (B, n_heads, d_k, N) → (B, n_heads, N, N)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = self.softmax(scores)
        attn = self.dropout(attn)
        # (B, n_heads, N, N) × (B, n_heads, N, d_k) → (B, n_heads, N, d_k)
        out = torch.matmul(attn, V)
        return out, attn

### 多头注意力机制（Pre-LN：不做残差和 Norm）

In [ ]:
class MultiHeadAttention(nn.Module):
    """Pre-LN 版本：不做残差连接，不做 LayerNorm，只做纯多头注意力计算"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

        self.attention = SelfAttention(dropout)
        self.dropout = nn.Dropout(dropout)
        # Pre-LN: 不再持有 LayerNorm，Norm 提到 EncoderLayer 里

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        # (B, N, d_model) → (B, N, n_heads, d_k) → (B, n_heads, N, d_k)
        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn = self.attention(Q, K, V, mask)

        # (B, n_heads, N, d_k) → (B, N, n_heads, d_k) → (B, N, d_model)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        out = self.fc(out)
        out = self.dropout(out)

        return out, attn  # Pre-LN: 不做 out+q，不做 norm

### 前馈神经网络（Pre-LN：不做残差和 Norm）

In [ ]:
class FeedForward(nn.Module):
    """Pre-LN 版本：不做残差连接，不做 LayerNorm，只做纯 MLP"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)   # 升维: d_model → 4*d_model
        self.fc2 = nn.Linear(d_ff, d_model)   # 降维: 4*d_model → d_model
        self.dropout = nn.Dropout(dropout)
        # Pre-LN: 不再持有 LayerNorm，Norm 提到 EncoderLayer 里

    def forward(self, x):
        # x: (B, N, d_model)  每个位置独立过 MLP
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))  # Pre-LN: 不做 out+x，不做 norm

## 2. ViT 新增组件

### Patch Embedding — 替代 nn.Embedding

Transformer 输入是 token ID → `nn.Embedding` 查表得到向量。  
ViT 输入是图像 → 把图像切成固定大小的 patch，每个 patch 展平后线性投影到 d_model。  
**一个 stride=patch_size 的 Conv2d 等价于 patchify + 线性投影。**

In [ ]:
class PatchEmbedding(nn.Module):
    """把 HxWxC 图像 → N 个 d_model 维的 patch token"""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2   # 例如 14×14 = 196

        # Conv2d 一步完成：滑动取 patch + 线性投影到 d_model
        # kernel_size=stride=patch_size → 不重叠地切块
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)          # (B, d_model, H/P, W/P)  例如 (B, 768, 14, 14)
        x = x.flatten(2)          # (B, d_model, N)          展平空间维度
        x = x.transpose(1, 2)    # (B, N, d_model)          变成序列格式
        return x

### EncoderLayer — Pre-LN 架构（ViT 论文原版）

对比 Post-LN（原版 Transformer）：

| | Post-LN（Transformer） | Pre-LN（ViT / GPT-2） |
|---|---|---|
| 顺序 | Sublayer → Add → **Norm** | **Norm** → Sublayer → Add |
| Norm 位置 | 在 MultiHeadAttention / FFN 内部 | 在 EncoderLayer 里，子层之前 |
| 训练稳定性 | 需要 warmup | 不需要 warmup，梯度更稳 |

ViT 不传 mask（没有 padding，没有因果遮罩）。

In [ ]:
class EncoderLayer(nn.Module):
    """Pre-LN 版本：LayerNorm 放在子层之前（ViT / GPT-2 的做法）"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)      # ← Norm 提到外面
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src, src_mask=None):
        # Pre-LN: Norm → Sublayer → Add
        x = src + self.self_attn(self.norm1(src), self.norm1(src), self.norm1(src), src_mask)[0]
        x = x + self.ffn(self.norm2(x))
        return x

### ViT — 核心模型

对比 Transformer 的 `Encoder` 类，ViT 替换了三件事：

| Transformer Encoder | ViT |
|---|---|
| `nn.Embedding(vocab_size, d_model)` | `PatchEmbedding(...)` |
| 正弦 `PositionEncoding` | 可学习 `nn.Parameter` |
| 无 CLS token | 可学习 `cls_token` prepend |

数据流：`图像 → PatchEmbedding → prepend CLS → + pos_embed → EncoderLayer×N → LayerNorm → CLS → 分类头`

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 n_classes=1000, d_model=768, n_heads=12, d_ff=3072,
                 num_layers=12, dropout=0.1):
        super().__init__()

        # ---- ① Patch Embedding（替代 nn.Embedding）----
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        n_patches = self.patch_embed.n_patches

        # ---- ② CLS token（可学习）----
        # (1, 1, d_model) → 广播到整个 batch
        # CLS token从所有patch收集信息，形成整张图的全局表示
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # ---- ③ 位置编码（可学习，替代正弦 PE）----
        # ViT 用可学习的 1D 位置编码，不是正弦函数
        # 长度 = 1(CLS) + n_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + n_patches, d_model))

        self.dropout = nn.Dropout(dropout)

        # ---- ④ Transformer Encoder 层堆叠（复用 EncoderLayer）----
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        # ---- ⑤ 最后的 LayerNorm + 分类头 ----
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

        # ---- 权重初始化 ----
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]

        # ① 图像 → patch tokens
        x = self.patch_embed(x)                      # (B, N, d_model)

        # ② 在最前面 prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, 1+N, d_model)

        # ③ 加上可学习位置编码（加法，不是拼接）
        x = x + self.pos_embed                        # (B, 1+N, d_model)
        x = self.dropout(x)

        # ④ 过 N 层 Transformer Encoder
        for layer in self.layers:
            x = layer(x, src_mask=None)               # ViT 不用 mask

        # ⑤ LayerNorm
        x = self.norm(x)

        # ⑥ 取 CLS token 对应的输出做分类
        cls_out = x[:, 0, :]                             # (B, d_model)
        return self.head(cls_out)                     # (B, n_classes)

## 3. Transformer → ViT 改造对照表

| 组件 | Transformer (`_Transformer.ipynb`) | ViT (`_vit.ipynb`) |
|---|---|---|
| `SelfAttention` | Post-LN | **Pre-LN 改造**（去掉内部 Norm 和残差） |
| `MultiHeadAttention` | 内部 `self.norm(out+q)` | **纯计算** `return out` |
| `FeedForward` | 内部 `self.norm(out+x)` | **纯计算** `return out` |
| `EncoderLayer` | Post-LN 透传 | **Pre-LN**：`norm1` → attn → add → `norm2` → ffn → add |
| Token 化 | `nn.Embedding(vocab, d_model)` + 查表 | `PatchEmbedding` — Conv2d 切图 + 投影 |
| 位置编码 | 正弦 `PositionEncoding`（固定，不可学） | `nn.Parameter` — 可学习 |
| CLS token | 无 | `nn.Parameter` — prepend 到序列最前 |
| 输出 | Decoder → `fc_out` → vocab_size | CLS → `head` → n_classes |
| `DecoderLayer` / `Decoder` | ✓ 需要 | **删除** |
| Mask | src_mask + tgt_mask + memory_mask | 全不需要 |

**一句话**：ViT = Transformer Encoder 的词嵌入换图块嵌入 + 正弦 PE 换可学习参数 + CLS token + Post-LN 改 Pre-LN，删掉 Decoder。